# Orca Core (Novus) — Probe-Grounded Safety DPO (Kaggle, on top of v2)

**Why this notebook exists**: `orca train redteam --model orca-core` shows a real,
unfixed 0.0% jailbreak block rate — every one of the 10 `JAILBREAK_PROBES` in
`orca/train/redteam.py` was complied with, none refused. This isn't a measurement
artifact (unlike the earlier accuracy-scoring instability this project hit) — it's
a real capability gap, confirmed across multiple redteam runs.

**Why DPO on the exact eval probes, not more SFT on synthetic ones**: an earlier
safety-training attempt for nano trained on teacher-invented adversarial prompts
(`generate_safety_refusal_pairs()` in `orca/train/dpo_pairs.py`) — stylistically
similar to, but literally different from, the fixed probes `redteam.py` measures
against. Train/eval mismatch. `generate_probe_grounded_safety_pairs()` closes that
gap by training directly on the exact probe text, with the teacher's refusal as
`chosen` and the weak (Ollama-served orca-core) model's actual compliant response
as `rejected` — captured with `trials=3`, only keeping a pair if the weak model
complied in every trial (a clean, unambiguous gap, not one trial's noise).

**What to upload as a Kaggle dataset before running this**:
- `core_probe_grounded_safety_dpo_<date>.jsonl` — generated locally via
  `generate_probe_grounded_safety_pairs(weak_model="orca-core", trials=3)`
- `core_v2_adapter/` — the adapter_config.json + adapter_model.safetensors from
  the orca-core-v2 SFT run (`orca_core_finetune_kaggle_v2.ipynb`'s output)

**Honest expectation-setting**: only 10 probes means at most 10 training pairs —
some will be dropped if the teacher didn't clearly refuse, or if orca-core's
compliance wasn't consistent across all 3 trials (see `generate_probe_grounded_safety_pairs`'s
own skip-reasons in its returned dict). This is a small, targeted correction on
top of an already-capable checkpoint, not a full retrain — DPO needs far fewer
examples and epochs than SFT for exactly this reason. If the resulting block rate
is still low after this, that's real signal the gap needs more/better pairs, not
a sign this approach is wrong.

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "0"

# transformers pinned <5 -- the unpinned install pulled transformers 5.x,
# whose new internal weight-conversion registry (core_model_loading.py)
# raises NotImplementedError inside model.save_pretrained_merged() /
# save_pretrained_gguf() for this architecture. Real bug hit on a live run,
# not a hypothetical -- training itself succeeded fine on 5.x, only the
# merge/export step broke. Pinning avoids it without guessing at unsloth internals.
!pip install -q "transformers<5" unsloth trl datasets peft bitsandbytes accelerate

## Find the uploaded preference-pair dataset and the v2 adapter

In [ ]:
import glob, json

safety_matches  = glob.glob('/kaggle/input/**/core_probe_grounded_safety_dpo*.jsonl', recursive=True)
adapter_matches = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)

print('Safety pairs file:', safety_matches)
print('Adapter config:', adapter_matches)

if not safety_matches or not adapter_matches:
    raise FileNotFoundError(
        "Upload core_probe_grounded_safety_dpo_<date>.jsonl and the core_v2 adapter "
        "folder (adapter_config.json + adapter_model.safetensors) as a Kaggle dataset "
        "and attach it to this notebook before running."
    )

safety_path = safety_matches[0]
adapter_dir = os.path.dirname(adapter_matches[0])

In [ ]:
def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    return records

safety_pairs = load_jsonl(safety_path)
print(f'safety pairs: {len(safety_pairs)}')
if len(safety_pairs) < 5:
    print('WARNING: fewer than 5 pairs — expected given only 10 probes exist, but '
          'double-check this is the file you meant to upload, not an empty/partial one.')

## Load base model (4-bit) + attach the v2 LoRA adapter (continuing from it, not from scratch)

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

max_seq_length = 2048
base_model = "unsloth/Meta-Llama-3.1-8B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Attach the v2 SFT adapter, trainable — DPO continues from where SFT left off
# instead of starting from a bare base model.
model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=True)

## Build the preference dataset in TRL's expected format

`{"prompt", "chosen", "rejected"}` — already the exact format
`generate_probe_grounded_safety_pairs()` wrote, no reformatting needed.

In [ ]:
from datasets import Dataset

dpo_ds = Dataset.from_list([
    {"prompt": r["prompt"], "chosen": r["chosen"], "rejected": r["rejected"]}
    for r in safety_pairs
])
print(f'dpo_ds = {len(dpo_ds)} pairs')

## DPO training

Beta 0.1 (standard), 3 epochs — with a dataset this small (at most 10 pairs),
more passes over the same examples is reasonable; still far less compute than
any SFT run. No mid-training checkpointing (a real Kaggle-side pickling bug hit
that path before) — the adapter-save cell right after training is the safety net.

In [ ]:
from trl import DPOConfig, DPOTrainer
import time

dpo_config = DPOConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    beta=0.1,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    save_strategy="no",
    output_dir="/kaggle/working/output",
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dpo_ds,
    processing_class=tokenizer,
)

start = time.time()
trainer.train()
print(f"[train] done in {time.time() - start:.0f}s")

## Save the DPO-updated adapter immediately (before merge/export)

In [ ]:
adapter_out_dir = "/kaggle/working/adapter_dpo"
model.save_pretrained(adapter_out_dir)
tokenizer.save_pretrained(adapter_out_dir)
print(f"[adapter] saved to {adapter_out_dir} — DPO-corrected weights are now safe on disk.")
!ls -la {adapter_out_dir}

## Merge LoRA + export GGUF (done in /tmp, not /kaggle/working)

Same disk-space fix as the SFT notebooks: `/kaggle/working/` has a 19.5GB quota
that a merged 16-bit 8B model + F16 GGUF intermediate would exceed.

In [ ]:
import shutil

shutil.rmtree("/tmp/merged", ignore_errors=True)
shutil.rmtree("/tmp/gguf", ignore_errors=True)

print("[merge] merging LoRA adapters (in /tmp)...")
model.save_pretrained_merged("/tmp/merged", tokenizer, save_method="merged_16bit")
print("[merge] saved to /tmp/merged")

print("[gguf] converting to GGUF q4_k_m (in /tmp)...")
try:
    model.save_pretrained_gguf("/tmp/gguf", tokenizer, quantization_method="q4_k_m")
    print("[gguf] saved under /tmp")
except Exception as e:
    # Real bug hit on a live run: deleting model.config.quantization_config
    # BEFORE this call (an earlier fix attempt) broke Unsloth's own LoRA
    # fusion logic -- it uses that attribute's presence to know it needs to
    # dequantize+fuse each layer, so removing it early causes it to skip
    # fusion entirely and dump raw PEFT-wrapped tensors instead. This retry
    # does NOT touch the live model. Instead: save_pretrained_gguf already
    # wrote a correctly-FUSED (but stale-config-tagged) HF checkpoint into
    # /tmp/gguf before it failed inside the llama.cpp subprocess call --
    # patch that already-written config.json and re-run just the convert
    # step ourselves, bypassing the wrapper only for this retry.
    print(f"[gguf] wrapper call failed ({e}); attempting manual recovery")
    import json as _json, os as _os, subprocess

    cfg_path = "/tmp/gguf/config.json"
    if not _os.path.exists(cfg_path):
        raise RuntimeError(
            "[gguf] no config.json found under /tmp/gguf -- the wrapper failed "
            "before writing the merged checkpoint at all, so there's nothing to "
            "recover here. See the original exception above."
        ) from e

    with open(cfg_path) as f:
        _cfg = _json.load(f)
    if "quantization_config" in _cfg:
        print("[gguf] removing stale quantization_config from /tmp/gguf/config.json")
        del _cfg["quantization_config"]
        with open(cfg_path, "w") as f:
            _json.dump(_cfg, f, indent=2)

    converter = "/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py"
    f16_out = "/tmp/gguf/model.F16.gguf"
    print(f"[gguf] manually re-running converter -> {f16_out}")
    subprocess.run(
        ["python3", converter, "--outfile", f16_out, "--outtype", "f16", "/tmp/gguf"],
        check=True,
    )
    print("[gguf] F16 conversion recovered successfully")

    # Quantize F16 -> Q4_K_M if the vendored llama-quantize binary is where
    # earlier error logs implied the rest of llama.cpp lives; if it's not
    # there under this exact name, this cell prints what it found instead
    # of failing blind, so the notebook run tells us the real layout rather
    # than guessing at a path a second time.
    import glob
    quant_bin_candidates = glob.glob("/root/.unsloth/llama.cpp/**/llama-quantize", recursive=True)
    print("[gguf] llama-quantize candidates found:", quant_bin_candidates)
    if quant_bin_candidates:
        q4_out = "/tmp/gguf/model.Q4_K_M.gguf"
        subprocess.run([quant_bin_candidates[0], f16_out, q4_out, "Q4_K_M"], check=True)
        print(f"[gguf] quantized to {q4_out}")
    else:
        print("[gguf] no llama-quantize binary found -- F16 GGUF at", f16_out,
              "is still usable directly (larger file, same behavior), "
              "quantize separately afterward if needed.")

In [ ]:
import glob, shutil, os

candidates = [f for f in glob.glob('/tmp/**/*.gguf', recursive=True) if 'q4_k_m' in f.lower()]
print('Found in /tmp:', candidates)

if candidates:
    source_path = candidates[0]
    filename = os.path.basename(source_path)
    dest_path = f'/kaggle/working/{filename}'
    shutil.copy(source_path, dest_path)
    print(f"[export] copied to {dest_path}")
    print("\nNext: click 'Save Version' -> 'Save & Run All (Commit)' at the top right.")
else:
    print('No GGUF file found — check the merge/export cells above for errors before committing.')